# План
1. Создать онтологию
2. Создать промпт
3. Построить граф на тексте
4. Код RAG
5. Промпт RAG
6. 

In [ ]:
# 0 утсановка ключей в env

In [26]:
%pip -q install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv("../.env")   # подхватит /.../neo4j_граф/.env

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", bool(DEEPSEEK_API_KEY))


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
DEEPSEEK_API_KEY loaded: True


In [27]:
# 1 стирание базы полное до нуля

In [28]:
def step_1_reset_db():
    !docker stop neo4j_travel || true
    !docker rm neo4j_travel || true
    !docker volume rm neo4j_travel_data || true
    
    !docker run -d --name neo4j_travel \
      -p 7475:7474 -p 7688:7687 \
      -e NEO4J_AUTH=neo4j/12345678 \
      -v neo4j_travel_data:/data \
      neo4j:5
    
    # !docker ps --filter name=neo4j_travel   
    print("Did reset DB")
    return None


In [29]:
# 2 сборка промпта и запрос к DeepSeek - выход ABox в виде JSON

In [39]:
def step_2_prompt_to_abox(msg_as_JSON) -> str:
    import os
    import json
    import requests
    import time

    PROMPT_PATH = "prompt_extractor.txt"
    ONTO_PATH = "ontology.txt"

    DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
    if not DEEPSEEK_API_KEY:
        raise RuntimeError("Set env var DEEPSEEK_API_KEY (export DEEPSEEK_API_KEY=...).")

    DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
    MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-reasoner")  # or deepseek-chat

    # --- load prompt template + ontology ---
    with open(PROMPT_PATH, "r", encoding="utf-8") as f:
        prompt_template = f.read()

    with open(ONTO_PATH, "r", encoding="utf-8") as f:
        ontology_text = f.read().strip()

    # --- normalize message JSON string ---
    if isinstance(msg_as_JSON, (dict, list)):
        message_json_string = json.dumps(msg_as_JSON, ensure_ascii=False)
    else:
        message_json_string = str(msg_as_JSON).strip()

    # validate that it's JSON (LLM expects JSON string with keys post_id/post_url/author/date/text)
    try:
        json.loads(message_json_string)
    except Exception as e:
        raise ValueError(f"msg_as_JSON must be a valid JSON string (or dict). Parse error: {e}") from e

    # --- stitch prompt ---
    prompt = (
        prompt_template
        .replace("{RDF_ONTOLOGY_HERE}", ontology_text)
        .replace("{MESSAGE_JSON_STRING_HERE}", message_json_string)
    )

    # sanity check for unresolved placeholders
    unresolved = [p for p in ["{RDF_ONTOLOGY_HERE}", "{MESSAGE_JSON_STRING_HERE}"] if p in prompt]
    if unresolved:
        raise RuntimeError(f"Unresolved placeholders in prompt: {unresolved}")

    # --- call DeepSeek ---
    url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL,
        "messages": [
            # system можно оставить очень коротким, промпт уже строгий
            {"role": "system", "content": "Return ONLY valid JSON object. No markdown. No code fences."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.0,
    }

    t0 = time.perf_counter()
    resp = requests.post(url, headers=headers, json=payload, timeout=180)
    elapsed = time.perf_counter() - t0
    print(f"DeepSeek API latency: {elapsed:.2f}s")
    resp.raise_for_status()
    data = resp.json()

    api_finish = data["choices"][0].get("finish_reason", "")
    content = data["choices"][0]["message"]["content"]

    emoji = "🟢" if api_finish != "length" else "🔴"
    print(f"{emoji} API finish_reason = {api_finish}")

    # optional: validate JSON (hard fail if model returned junk)
    try:
        json.loads(content)
    except Exception as e:
        raise ValueError(f"Model did not return valid JSON. Error: {e}\nRaw content:\n{content[:2000]}") from e

    print(content)  # preview
    return content


In [31]:
# 3 JSON → Cypher генерация

In [32]:
from typing import List

def step_4_abox_to_cypher(LLM_answer: str) -> List[str]:
    import json
    import re

    # =========================
    # Helpers
    # =========================

    def strip_code_fences(s: str) -> str:
        s = (s or "").strip()
        if s.startswith("```"):
            s = re.sub(r"^```(?:json)?\s*", "", s)
            s = re.sub(r"\s*```$", "", s)
        return s.strip()

    def neo4j_local_name(name: str) -> str:
        if not name:
            return ""
        # ex:entityName -> entityName
        if ":" in name:
            return name.split(":")[-1]
        if "#" in name:
            return name.split("#")[-1]
        if "/" in name:
            return name.rsplit("/", 1)[-1]
        return name

    def neo4j_safe_ident(name: str) -> str:
        """
        Neo4j-safe identifier for labels / rel-types / property keys.
        Keep ASCII letters/digits/_ ; everything else -> _
        """
        n = neo4j_local_name(name)
        n = re.sub(r"[^A-Za-z0-9_]", "_", n)
        if not n:
            return "X"
        if n[0].isdigit():
            n = "_" + n
        return n

    def contains_cyrillic(s: str) -> bool:
        return bool(re.search(r"[А-Яа-яЁё]", s or ""))

    def cypher_literal(v):
        if v is None:
            return "null"
        if isinstance(v, bool):
            return "true" if v else "false"
        if isinstance(v, (int, float)):
            return str(v)

        # strings
        s = str(v)
        s = s.replace("\\", "\\\\").replace("'", "\\'")
        s = s.replace("\n", "\\n").replace("\r", "\\r")
        return f"'{s}'"

    # =========================
    # Parse JSON from LLM
    # =========================

    raw = strip_code_fences(LLM_answer)
    root = json.loads(raw)

    payload = root.get("payload", {}) or {}
    nodes = payload.get("nodes", []) or []
    rels = payload.get("relationships", []) or []

    cypher_statements: List[str] = []

    # =========================
    # Constraints (unique key per label)
    # =========================

    labels = sorted({
        neo4j_safe_ident(n.get("label", ""))
        for n in nodes
        if n.get("label")
    })

    for label in labels:
        cypher_statements.append(
            f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{label}) REQUIRE n.key IS UNIQUE"
        )

    # =========================
    # Nodes (MERGE by key, then SET props)
    # =========================

    for node in nodes:
        label_raw = node.get("label", "")
        key = node.get("key")

        if not label_raw or key is None:
            continue

        label = neo4j_safe_ident(label_raw)
        props = node.get("properties", {}) or {}

        set_parts = []
        for k, v in props.items():
            prop_key = neo4j_safe_ident(k)

            if isinstance(v, str) and contains_cyrillic(v):
                print(f"⚠ WARNING: Cyrillic detected in property '{prop_key}' → '{v}'")

            set_parts.append(f"n.{prop_key}={cypher_literal(v)}")

        set_clause = (" SET " + ", ".join(set_parts)) if set_parts else ""
        cypher_statements.append(
            f"MERGE (n:{label} {{key:{cypher_literal(key)}}}){set_clause}"
        )

    # =========================
    # Relationships
    # =========================

    for rel in rels:
        f = rel.get("from", {}) or {}
        t = rel.get("to", {}) or {}

        fl_raw = f.get("label", "")
        fk = f.get("key")
        tl_raw = t.get("label", "")
        tk = t.get("key")
        rt_raw = rel.get("type", "")

        if not fl_raw or not tl_raw or not rt_raw or fk is None or tk is None:
            continue

        fl = neo4j_safe_ident(fl_raw)
        tl = neo4j_safe_ident(tl_raw)
        rt = neo4j_safe_ident(rt_raw)

        cypher_statements.append(
            f"MATCH (a:{fl} {{key:{cypher_literal(fk)}}}), "
            f"(b:{tl} {{key:{cypher_literal(tk)}}}) "
            f"MERGE (a)-[:{rt}]->(b)"
        )

    # =========================
    # Output
    # =========================

    print("LLM finish_reason:", root.get("finish_reason"))
    print("LLM reason:", root.get("reason"))
    print(f"Prepared {len(cypher_statements)} Cypher statements.\n")

    return cypher_statements

In [33]:
# 4 Сохранение Cypher в чекпоинт

In [34]:
def step_5_save_cypher(cypher_statements: List[str]):
    import os
    import re
    
    ckpt_dir = "cypher_checkpoints"
    os.makedirs(ckpt_dir, exist_ok=True)
    
    existing = []
    for name in os.listdir(ckpt_dir):
        m = re.match(r"^cypher_(\d+)\.txt$", name)
        if m:
            existing.append(int(m.group(1)))
    
    next_num = max(existing, default=0) + 1
    out_path = os.path.join(ckpt_dir, f"cypher_{next_num}.txt")
    
    with open(out_path, "w", encoding="utf-8") as f:
        for stmt in cypher_statements:
            f.write(stmt)
            f.write(";")
    
    print(f"Saved {len(cypher_statements)} statements to {out_path}")
    return None


In [35]:
# 5 исполнение Cypher (каждая строка автономна)

In [36]:
def step_6_execute_cypher(cypher_statements: List[str]):
    from neo4j import GraphDatabase
    
    URI = "neo4j://localhost:7688"
    USER = "neo4j"
    PASSWORD = "12345678"
    DATABASE = "neo4j"
    
    driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
    
    try:
        with driver.session(database=DATABASE) as session:
            for i, stmt in enumerate(cypher_statements, 1):
                session.run(stmt).consume()
        print(f"✅ Executed {len(cypher_statements)} statements.")
    finally:
        driver.close()    
    return None


In [37]:
# 7 запуск пайплайна

In [40]:
step_1_reset_db()

import json
import subprocess


with open("messages.json", "r", encoding="utf-8") as f:
    messages = json.load(f)
messages = list(map(lambda x: json.dumps(x, ensure_ascii=False), messages[:1])) # берем первое и только его


for msg in messages:
    print(msg)
    abox_json = step_2_prompt_to_abox(msg) 
    cypher_statements = step_4_abox_to_cypher(abox_json)
    step_5_save_cypher(cypher_statements)
    step_6_execute_cypher(cypher_statements)

subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)


neo4j_travel
neo4j_travel
neo4j_travel_data
6d4d2d6e84585152e0db61cbd3e5d708286b03095b39b3141bb9cdca38bb57e6
Did reset DB
{"post_id": 77777777, "post_url": "https://vk.com", "author": "IVAN_SHARGANOV", "date": "11 июн 2013, 09:05", "text": "Elon Musk is the CEO of SpaceX. SpaceX was founded in 2002. Elon Musk was born in South Africa. Elon Musk’s email is elon@spacex.com."}
DeepSeek API latency: 77.72s
🟢 API finish_reason = stop
{
  "payload": {
    "ontology_version": "https://example.org/elon#",
    "nodes": [
      {
        "label": "Person",
        "key": "person_elon_musk",
        "properties": {
          "entityName": "Elon Musk",
          "email": "elon@spacex.com"
        }
      },
      {
        "label": "Company",
        "key": "company_spacex",
        "properties": {
          "entityName": "SpaceX",
          "foundedInYear": 2002
        }
      },
      {
        "label": "Country",
        "key": "country_south_africa",
        "properties": {
          "entityN

NameError: name 'subprocess' is not defined

In [41]:
# 8 RAG

In [89]:
from pathlib import Path
import json, os, re, time
from datetime import datetime
import requests
from collections import Counter

BASE_DIR = Path("/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/RAG_elon_playground")

DATA_PATH = BASE_DIR / "gold_retrieval.jsonl"
SCHEMA_PATH = BASE_DIR / "ontology.txt"
PROMPT_RETRIEVAL_PATH = BASE_DIR / "prompt_retrieval.txt"
PROMPT_OPTIMIZER_PATH = BASE_DIR / "prompt_optimizer.txt"
HISTORY_DIR = BASE_DIR / "prompt_history"
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def pick_splits(rows):
    holdout = [r for r in rows if "holdout" in r.get("id", "").lower()]
    if holdout:
        test = holdout[0]
        rest = [r for r in rows if r is not test]
    else:
        test = rows[-1]
        rest = rows[:-1]

    val = None
    train = rest
    if len(rest) >= 3:
        val = rest[-1]
        train = rest[:-1]
    return train, val, test

def strip_code_fences(s: str) -> str:
    s = (s or "").strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def normalize_cypher(q: str) -> str:
    q = strip_code_fences(q)
    q = re.sub(r"/\*.*?\*/", "", q, flags=re.S)
    q = re.sub(r"//.*?$", "", q, flags=re.M)
    q = q.replace("`", "").replace('"', "'")
    q = re.sub(r"\s+", " ", q).strip()
    q = re.sub(r";$", "", q)
    return q.lower()

def exact_match(a: str, b: str) -> bool:
    return normalize_cypher(a) == normalize_cypher(b)

def token_f1(a: str, b: str) -> float:
    ta = normalize_cypher(a).split()
    tb = normalize_cypher(b).split()
    if not ta and not tb:
        return 1.0
    if not ta or not tb:
        return 0.0
    ca, cb = Counter(ta), Counter(tb)
    overlap = sum((ca & cb).values())
    precision = overlap / len(ta)
    recall = overlap / len(tb)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def deepseek_chat(messages, temperature=0.0, max_tokens=256, model="deepseek-chat"):
    api_key = os.getenv("DEEPSEEK_API_KEY")
    if not api_key:
        raise RuntimeError("Set env var DEEPSEEK_API_KEY before running.")
    base_url = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    t0 = time.perf_counter()
    resp = requests.post(f"{base_url}/v1/chat/completions", headers=headers, json=payload, timeout=180)
    dt = time.perf_counter() - t0
    resp.raise_for_status()
    data = resp.json()
    finish = data["choices"][0].get("finish_reason", "")
    content = data["choices"][0]["message"]["content"]
    print(f"LLM latency: {dt:.2f}s, finish={finish}")
    return content, finish

def call_llm_with_retry(messages, temperature=0.0, max_tokens=256, model="deepseek-chat", retries=1):
    last = ""
    for attempt in range(retries + 1):
        content, finish = deepseek_chat(messages, temperature=temperature, max_tokens=max_tokens, model=model)
        last = content
        if content.strip() and finish != "length":
            return content
        if attempt < retries:
            messages = messages + [{"role": "user", "content": "Short answer only. One line. No reasoning."}]
    return last

def extract_cypher(text: str) -> str:
    text = strip_code_fences(text)
    if not text:
        return ""
    m = re.search(r"(match\\s+.*)", text, flags=re.I | re.S)
    return m.group(1).strip() if m else text.strip()

def generate_cypher(prompt_text: str, schema_text: str, question: str) -> str:
    prompt = (
        prompt_text.replace("{SCHEMA}", schema_text)
                   .replace("{QUESTION}", question)
    )
    messages = [
        {"role": "system", "content": "Return ONLY Cypher. No markdown. No explanations. No reasoning."},
        {"role": "user", "content": prompt},
    ]
    content = call_llm_with_retry(messages, temperature=0.0, max_tokens=128, model="deepseek-chat", retries=1)
    return extract_cypher(content)

# def propose_prompt_update(optimizer_prompt_text: str, current_prompt: str, schema_text: str,
#                           question: str, gold: str, pred: str) -> str:
#     prompt = optimizer_prompt_text
#     prompt = prompt.replace("{CURRENT_PROMPT}", current_prompt)
#     prompt = prompt.replace("{SCHEMA}", schema_text)
#     prompt = prompt.replace("{QUESTION}", question)
#     prompt = prompt.replace("{GOLD_CYPHER}", gold)
#     prompt = prompt.replace("{PRED_CYPHER}", pred)

#     messages = [
#         {"role": "system", "content": "Return ONLY the full revised prompt text. Keep placeholders and markers."},
#         {"role": "user", "content": prompt},
#     ]
#     return strip_code_fences(call_llm_with_retry(messages, temperature=0.2, max_tokens=900, model="deepseek-chat", retries=1))

# def validate_prompt_text(p: str) -> bool:
#     return all(k in p for k in ["{SCHEMA}", "{QUESTION}", "BEGIN_RULES", "END_RULES"])

def split_prompt(prompt_text: str):
    m = re.search(r"(.*)BEGIN_RULES(.*)END_RULES(.*)", prompt_text, flags=re.S)
    if not m:
        raise ValueError("BEGIN_RULES/END_RULES not found in prompt.")
    prefix, rules, suffix = m.group(1), m.group(2), m.group(3)
    return prefix, rules.strip("\n"), suffix

def build_prompt(prefix: str, rules: str, suffix: str) -> str:
    # гарантируем, что внешние блоки остаются неизменными
    return f"{prefix}BEGIN_RULES\n{rules}\nEND_RULES{suffix}"

def extract_rules_only(text: str) -> str:
    text = strip_code_fences(text)
    m = re.search(r"BEGIN_RULES(.*)END_RULES", text, flags=re.S)
    if m:
        return m.group(1).strip("\n")
    return text.strip()

def validate_prompt_text(p: str) -> bool:
    return all(k in p for k in ["{SCHEMA}", "{QUESTION}", "BEGIN_RULES", "END_RULES"])

def propose_prompt_update(optimizer_prompt_text: str, current_prompt: str, schema_text: str,
                          question: str, gold: str, pred: str) -> str:
    prefix, current_rules, suffix = split_prompt(current_prompt)

    prompt = optimizer_prompt_text
    prompt = prompt.replace("{CURRENT_RULES}", current_rules)
    prompt = prompt.replace("{SCHEMA}", schema_text)
    prompt = prompt.replace("{QUESTION}", question)
    prompt = prompt.replace("{GOLD_CYPHER}", gold)
    prompt = prompt.replace("{PRED_CYPHER}", pred)

    messages = [
        {"role": "system", "content": "Return ONLY the revised rules block text. No markers. No explanations."},
        {"role": "user", "content": prompt},
    ]
    candidate_raw = call_llm_with_retry(messages, temperature=0.2, max_tokens=700, model="deepseek-chat", retries=1)
    candidate_rules = extract_rules_only(candidate_raw)

    if not candidate_rules.strip():
        return current_prompt  # страховка: не меняем

    return build_prompt(prefix, candidate_rules, suffix)


def save_prompt_version(prompt_text: str, tag: str) -> Path:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = HISTORY_DIR / f"prompt_retrieval_{tag}_{ts}.txt"
    path.write_text(prompt_text, encoding="utf-8")
    return path

def eval_prompt(prompt_text: str, dataset, schema_text: str):
    if not dataset:
        return 0.0, 0.0, []
    results = []
    for ex in dataset:
        pred = generate_cypher(prompt_text, schema_text, ex["question"])
        results.append({
            "id": ex["id"],
            "question": ex["question"],
            "pred": pred,
            "gold": ex["gold_cypher"],
            "exact": exact_match(pred, ex["gold_cypher"]),
            "f1": token_f1(pred, ex["gold_cypher"]),
        })
    exact = sum(r["exact"] for r in results) / len(results)
    f1 = sum(r["f1"] for r in results) / len(results)
    return exact, f1, results

rows = load_jsonl(DATA_PATH)
train, val, test = pick_splits(rows)

schema_text = SCHEMA_PATH.read_text(encoding="utf-8").strip()
retrieval_prompt = PROMPT_RETRIEVAL_PATH.read_text(encoding="utf-8")
optimizer_prompt = PROMPT_OPTIMIZER_PATH.read_text(encoding="utf-8")

print(f"Train: {len(train)}, Val: {0 if val is None else 1}, Test: 1")
save_prompt_version(retrieval_prompt, "init")

best_val_exact, best_val_f1 = 0.0, 0.0
if val is not None:
    best_val_exact, best_val_f1, _ = eval_prompt(retrieval_prompt, [val], schema_text)
    print(f"Initial val exact={best_val_exact:.2f}, f1={best_val_f1:.2f}")

for i, ex in enumerate(train, 1):
    pred = generate_cypher(retrieval_prompt, schema_text, ex["question"])
    is_match = exact_match(pred, ex["gold_cypher"])
    print(f"[train {i}/{len(train)}] id={ex['id']} exact={is_match}")

    if is_match:
        continue

    candidate = propose_prompt_update(
        optimizer_prompt, retrieval_prompt, schema_text,
        ex["question"], ex["gold_cypher"], pred
    )
    if not validate_prompt_text(candidate):
        print("Candidate rejected: missing placeholders or markers")
        continue

    if val is not None:
        cand_exact, cand_f1, _ = eval_prompt(candidate, [val], schema_text)
        if (cand_exact < best_val_exact) or (cand_exact == best_val_exact and cand_f1 < best_val_f1):
            print(f"Candidate rejected: val did not improve (exact={cand_exact:.2f}, f1={cand_f1:.2f})")
            continue
        best_val_exact, best_val_f1 = cand_exact, cand_f1

    retrieval_prompt = candidate
    save_prompt_version(retrieval_prompt, f"upd_{i}")
    print("Prompt updated.")

PROMPT_RETRIEVAL_PATH.write_text(retrieval_prompt, encoding="utf-8")

train_exact, train_f1, _ = eval_prompt(retrieval_prompt, train, schema_text)
test_exact, test_f1, test_rows = eval_prompt(retrieval_prompt, [test], schema_text)

print(f"Final train exact={train_exact:.2f}, f1={train_f1:.2f}")
print(f"Final test  exact={test_exact:.2f}, f1={test_f1:.2f}")
print("Test detail:")
print(json.dumps(test_rows, ensure_ascii=False, indent=2))


Train: 4, Val: 1, Test: 1
LLM latency: 2.04s, finish=stop
Initial val exact=0.00, f1=0.71
LLM latency: 1.62s, finish=stop
[train 1/4] id=ex_company_founded exact=False
LLM latency: 5.12s, finish=stop
LLM latency: 2.46s, finish=stop
Prompt updated.
LLM latency: 2.41s, finish=stop
[train 2/4] id=ex_birth_country exact=False
LLM latency: 5.83s, finish=stop
LLM latency: 2.30s, finish=stop
Prompt updated.
LLM latency: 1.74s, finish=stop
[train 3/4] id=ex_person_email exact=True
LLM latency: 2.04s, finish=stop
[train 4/4] id=ex_all_people exact=False
LLM latency: 6.57s, finish=stop
LLM latency: 2.34s, finish=stop
Prompt updated.
LLM latency: 2.10s, finish=stop
LLM latency: 2.09s, finish=stop
LLM latency: 1.57s, finish=stop
LLM latency: 2.16s, finish=stop
LLM latency: 1.71s, finish=stop
Final train exact=0.50, f1=0.87
Final test  exact=0.00, f1=0.78
Test detail:
[
  {
    "id": "ex_holdout_ceo",
    "question": "Who is the CEO of SpaceX?",
    "pred": "MATCH (p:Person)-[:CEO_OF]->(c:Company {